In [ ]:
import SimpleITK as sitk
import numpy as np
import os

def bounding_box(colon_mask):

   # Convert the colon mask to a numpy array
    colon_array = sitk.GetArrayFromImage(colon_mask)  # Shape: [Z, Y, X]
    print("colon_mask.GetSize():", colon_mask.GetSize()) 
    print("colon_array.shape:", colon_array.shape)             # ?? (zSlices, ySize, xSize)
    print("colon_mask.GetSpacing():", colon_mask.GetSpacing()) # (spacingX, spacingY, spacingZ)
    print("colon_mask.GetDirection():", colon_mask.GetDirection()) 

    # Find the indices where the colon mask is non-zero
    colon_indices = np.nonzero(colon_array)

    # Extract the Z coordinates
    Z_coords = colon_indices[0]
    min_Z_colon = np.min(Z_coords)
    print("min_Z_colon", min_Z_colon)
    max_Z_rectum = min_Z_colon

    # Get the spacing in the Z-direction (slice thickness)
    voxel_height = colon_mask.GetSpacing()[2]  # In millimeters  # Shape: [X, Y, Z]

    # Define the physical height of the rectum area (e.g., 100 mm)
    rectum_height_mm = 50  # Adjust based on anatomical knowledge

    # Calculate the number of slices to include
    num_voxels = int(rectum_height_mm / voxel_height) #here voxel_height = 0.43

    # Define the upper Z limit for the rectum area
    print("num_voxels", num_voxels)
    
    min_Z_rectum = min_Z_colon - num_voxels
    print("rectum_min_Z", min_Z_rectum)
    
    min_Z_rectum = max(min_Z_rectum, 0)  # Ensure within bounds
    print("rectum_min_Z2", min_Z_rectum)
    
    # Initialize an array to hold the rectum mask
    rectum_array = np.zeros_like(colon_array)

    # Copy the relevant slices from the colon mask
    rectum_array[min_Z_rectum:, :, :] = colon_array[min_Z:rectum_min_Z, :, :]

    # # Convert the numpy array back to a SimpleITK image
    # rectum_mask = sitk.GetImageFromArray(rectum_array)

    # # Copy spatial information from the original colon mask
    # rectum_mask.CopyInformation(colon_mask)

    # Find the non-zero indices in the rectum mask
    rectum_indices = np.nonzero(rectum_array)
    print('0',np.count_nonzero(rectum_array))

    # Get the minimum and maximum indices for each dimension
    min_Z_rect, max_Z_rect = np.min(rectum_indices[0]), np.max(rectum_indices[0])
    min_Y_rect, max_Y_rect = np.min(rectum_indices[1]), np.max(rectum_indices[1])
    min_X_rect, max_X_rect = np.min(rectum_indices[2]), np.max(rectum_indices[2])
    print("min_Z1", min_Z_rect)           
    print("max_Z1", max_Z_rect)
    print("min_Y1", min_Y_rect) 
    print("max_Y1", max_Y_rect)           
    print("min_X1", min_X_rect)
    print("max_X1", max_X_rect)

    
    # Define the expansion size (in voxels)
    expansion = 10  # Adjust based on desired margin

    # Get the dimensions of the image
    image_size = colon_mask.GetSize()  # Returns (X, Y, Z)

    # Expand the bounding box, ensuring it stays within image bounds
    min_Z = max(min_Z_rect - expansion, 0)
    max_Z = min(max_Z_rect + expansion, image_size[2] - 1)

    min_Y = max(min_Y_rect - expansion, 0)
    max_Y = min(max_Y_rect + expansion, image_size[1] - 1)

    min_X = max(min_X_rect - expansion, 0)
    max_X = min(max_X_rect + expansion, image_size[0] - 1)

    print("min_Z", min_Z)           
    print("max_Z", max_Z)
    print("min_Y", min_Y) 
    print("max_Y", max_Y)           
    print("min_X", min_X)
    print("max_X", max_X)
    
    return min_Z, max_Z, min_Y, max_Y, min_X, max_X

In [ ]:
def extract_patches(minZ, maxZ, minY, maxY, minX, maxX, mr_image):
    # Define the size and index for the Region of Interest (ROI)
    print("min_Z", minZ)           
    print("max_Z", maxZ)
    print("min_Y", minY) 
    print("max_Y", maxY)           
    print("min_X", minX)
    print("max_X", maxX)

    roi_size = [
        int(maxX - minX + 1),
        int(maxY - minY + 1),
        int(maxZ - minZ + 1)
    ]

    roi_index = [
        int(minX),
        int(minY),
        int(minZ)
    ]

    # Set up the RegionOfInterestImageFilter
    roi_filter = sitk.RegionOfInterestImageFilter() #(X,Y,Z)
    roi_filter.SetSize(roi_size)
    roi_filter.SetIndex(roi_index)

    # Extract the patch
    rectum_patch = roi_filter.Execute(mr_image)
    return rectum_patch

In [ ]:
# Load the original MR image
path = r'/path/to/workspace/classificationmodel_original_sagittal/total_work/resampled'
save_path = r'/path/to/workspace/classificationmodel_original_sagittal/total_work/patches_all'

for i in os.listdir(path)[:2]:
    foldername = i # patient name
    file_path = os.path.join(path, i)
    image_path =  os.path.join(file_path, 'resampled_image.nii.gz')
    colon_path =  os.path.join(file_path, 'colon.nii.gz')

    image = sitk.ReadImage(image_path)
    colon_img = sitk.ReadImage(colon_path)

    try:
        min_Z, max_Z, min_Y, max_Y, min_X, max_X = bounding_box(colon_img)
        rectum_patch = extract_patches(min_Z, max_Z, min_Y, max_Y, min_X, max_X, image)
    
        # Save the extracted patch
        new_foldername = foldername + '.nii.gz'
        save_path_new = os.path.join(save_path, new_foldername)
        sitk.WriteImage(rectum_patch, save_path_new)
        
    except Exception as e:
        print(f"An error occurred: {e}")
        print("patient number:", foldername)
        
    
print('done')